# Wave Equation Toy Lab — 8 expériences progressives

[🇫🇷 Français](../../docs/waves-toy-lab.md#fr) · [🇬🇧 English](../../docs/waves-toy-lab.md#en) · [🇪🇸 Español](../../docs/waves-toy-lab.md#es) · [🇵🇹 Português](../../docs/waves-toy-lab.md#pt)

**Règle du jeu : prédire avant d'exécuter.** Pour chaque expérience, écrire mentalement ce qui devrait se passer, puis comparer la simulation à cette prédiction. Le guide détaillé relit chaque formule en français courant.

Ce notebook utilise l'équation d'onde scalaire, pas un modèle complet de fluide.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (root / 'src').exists():
        sys.path.insert(0, str(root / 'src'))
        break

from diderot_mls.waves_fdtd import (
    cfl_1d, cfl_2d, gaussian_pulse_1d, mode_shape_neumann_2d,
    mode_angular_frequency, point_source_1d, point_source_2d,
    step_1d, step_2d,
)

def animate_1d(x, frames, title='', ylim=None, interval=40):
    fig, ax = plt.subplots(figsize=(8, 3))
    line, = ax.plot(x, frames[0])
    ax.set_xlim(x[0], x[-1]); ax.set_xlabel('x'); ax.set_ylabel('u')
    ax.set_title(title)
    if ylim is None:
        m = max(1e-9, max(float(np.max(np.abs(f))) for f in frames))
        ylim = (-1.1*m, 1.1*m)
    ax.set_ylim(*ylim)
    def update(i):
        line.set_ydata(frames[i]); return (line,)
    ani = FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())

def animate_2d(frames, title='', extent=(0,1,0,1), interval=50):
    fig, ax = plt.subplots(figsize=(5, 5))
    m = max(1e-9, max(float(np.max(np.abs(f))) for f in frames))
    im = ax.imshow(frames[0].T, origin='lower', extent=extent,
                   cmap='RdBu_r', vmin=-m, vmax=m, animated=True)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title(title)
    fig.colorbar(im, ax=ax, label='u')
    def update(i):
        im.set_data(frames[i].T); return (im,)
    ani = FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())

def run_1d(u0, x, *, c=1.0, dt=None, nsteps=400, source_fn=None, boundary='neumann', save_every=4):
    dx = x[1]-x[0]
    if dt is None: dt = 0.85*dx/c
    print('CFL 1D:', cfl_1d(c, dt, dx))
    u_prev = u0.copy(); u = u0.copy(); frames=[u.copy()]
    for k in range(nsteps):
        t = k*dt
        src = None if source_fn is None else source_fn(t)
        u_next = step_1d(u_prev, u, c=c, dt=dt, dx=dx, source=src, boundary=boundary)
        u_prev, u = u, u_next
        if (k+1) % save_every == 0: frames.append(u.copy())
    return frames, dt

def run_2d(u0, x, y, *, c=1.0, dt=None, nsteps=300, source_fn=None, boundary='neumann', save_every=4):
    dx=x[1]-x[0]; dy=y[1]-y[0]
    if dt is None: dt=0.90/(c*np.sqrt(1/dx**2+1/dy**2))
    print('CFL 2D:', cfl_2d(c, dt, dx, dy))
    u_prev=u0.copy(); u=u0.copy(); frames=[u.copy()]
    for k in range(nsteps):
        t=k*dt
        src=None if source_fn is None else source_fn(t)
        u_next=step_2d(u_prev,u,c=c,dt=dt,dx=dx,dy=dy,source=src,boundary=boundary)
        u_prev,u=u,u_next
        if (k+1)%save_every==0: frames.append(u.copy())
    return frames, dt

## 1 — Impulsion 1D
On crée une bosse gaussienne et on la relâche avec une vitesse initiale nulle. **Prédiction :** elle doit se séparer en deux ondes qui partent en sens opposés.

In [ ]:
x=np.linspace(0,1,401)
u0=gaussian_pulse_1d(x, center=0.5, sigma=0.035)
frames,_=run_1d(u0,x,nsteps=170,boundary='neumann',save_every=2)
display(animate_1d(x,frames,'Exp. 1 — impulsion 1D'))

## 2 — Source sinusoïdale 1D
Une source localisée oscille comme `sin(ωt)`. **Prédiction :** un train d'ondes périodique quitte la source vers les deux côtés ; si la fréquence augmente à vitesse `c` fixée, la longueur d'onde diminue.

In [ ]:
x=np.linspace(0,1,401); i0=len(x)//2; omega=2*np.pi*12
def src2(t): return point_source_1d(len(x),i0,6000*np.sin(omega*t))
frames,_=run_1d(np.zeros_like(x),x,nsteps=220,source_fn=src2,boundary='neumann',save_every=2)
display(animate_1d(x,frames,'Exp. 2 — source harmonique 1D'))

## 3 — Deux sources 1D et déphasage
On additionne deux sources cohérentes. Testez `phi=0`, puis `phi=np.pi`. **Prédiction :** le changement de phase modifie fortement les zones de renforcement et d'annulation.

In [ ]:
x=np.linspace(0,1,401); i1=int(.35*len(x)); i2=int(.65*len(x)); omega=2*np.pi*10; phi=0.0
def src3(t):
    return (point_source_1d(len(x),i1,5000*np.sin(omega*t)) +
            point_source_1d(len(x),i2,5000*np.sin(omega*t+phi)))
frames,_=run_1d(np.zeros_like(x),x,nsteps=240,source_fn=src3,boundary='neumann',save_every=2)
display(animate_1d(x,frames,f'Exp. 3 — deux sources, phi={phi:.2f} rad'))

## 4 — Une source ponctuelle en 2D
Dans un milieu homogène, aucune direction n'est privilégiée. **Prédiction :** les fronts d'onde sont approximativement circulaires. La grille carrée peut toutefois laisser une petite signature numérique.

In [ ]:
x=np.linspace(0,1,121); y=np.linspace(0,1,121); shape=(len(x),len(y)); omega=2*np.pi*8
ij=(len(x)//2,len(y)//2)
def src4(t): return point_source_2d(shape,ij,12000*np.sin(omega*t))
frames,_=run_2d(np.zeros(shape),x,y,nsteps=150,source_fn=src4,boundary='neumann',save_every=3)
display(animate_2d(frames,'Exp. 4 — source ponctuelle 2D'))

## 5 — Deux sources 2D : franges d'interférence
Deux sources de même fréquence produisent des zones constructives et destructives. **Prédiction :** les franges se déplacent ou changent de forme si l'on modifie la distance, la fréquence ou la phase.

In [ ]:
x=np.linspace(0,1,141); y=np.linspace(0,1,141); shape=(len(x),len(y)); omega=2*np.pi*9; phi=0.0
a=(int(.35*len(x)),len(y)//2); b=(int(.65*len(x)),len(y)//2)
def src5(t):
    return (point_source_2d(shape,a,12000*np.sin(omega*t)) +
            point_source_2d(shape,b,12000*np.sin(omega*t+phi)))
frames,_=run_2d(np.zeros(shape),x,y,nsteps=180,source_fn=src5,boundary='neumann',save_every=3)
display(animate_2d(frames,'Exp. 5 — interférence de deux sources'))

## 6 — Réflexion : une source peut interférer avec ses propres réflexions
On conserve des bords de Neumann réfléchissants et on laisse la simulation durer davantage. **Point conceptuel :** un motif complexe ne signifie pas nécessairement qu'il existe plusieurs sources physiques.

In [ ]:
x=np.linspace(0,1,121); y=np.linspace(0,1,121); shape=(len(x),len(y)); omega=2*np.pi*7
ij=(int(.22*len(x)),len(y)//2)
def src6(t): return point_source_2d(shape,ij,10000*np.sin(omega*t))
frames,_=run_2d(np.zeros(shape),x,y,nsteps=330,source_fn=src6,boundary='neumann',save_every=4)
display(animate_2d(frames,'Exp. 6 — réflexions sur les bords'))

## 7 — Mode propre d'une cavité rectangulaire
Pour des murs de Neumann, `cos(mπx/Lx) cos(nπy/Ly)` est une forme propre. **Prédiction :** la géométrie du motif reste reconnaissable tandis que son amplitude oscille dans le temps.

In [ ]:
x=np.linspace(0,1,121); y=np.linspace(0,1,121); m,n=2,3
u0=mode_shape_neumann_2d(x,y,m=m,n=n,lx=1,ly=1)
print('omega_mn =',mode_angular_frequency(1.0,m=m,n=n,lx=1,ly=1))
frames,_=run_2d(u0,x,y,nsteps=220,boundary='neumann',save_every=3)
display(animate_2d(frames,f'Exp. 7 — mode propre ({m},{n})'))

## 8 — Casser volontairement la condition CFL
Sur une grille carrée 2D, le schéma classique exige `c*dt/h <= 1/sqrt(2)`. On compare un pas stable à un pas trop grand. **Prédiction :** dans le second cas, de minuscules erreurs numériques sont amplifiées jusqu'à dominer la solution.

In [ ]:
x=np.linspace(0,1,81); y=np.linspace(0,1,81); h=x[1]-x[0]; c=1.0
xx,yy=np.meshgrid(x,y,indexing='ij')
u0=np.exp(-((xx-.5)**2+(yy-.5)**2)/(2*.04**2))
def growth(dt,nsteps=140):
    report=cfl_2d(c,dt,h,h); print(report)
    up=u0.copy(); u=u0.copy(); maxima=[]
    for _ in range(nsteps):
        un=step_2d(up,u,c=c,dt=dt,dx=h,dy=h,boundary='neumann')
        up,u=u,un; maxima.append(float(np.max(np.abs(u))))
    return np.array(maxima)
stable_dt=.65*h/c
unstable_dt=.80*h/c
g1=growth(stable_dt); g2=growth(unstable_dt)
plt.figure(figsize=(8,4)); plt.semilogy(g1,label='stable: c dt/h = 0.65'); plt.semilogy(g2,label='unstable: c dt/h = 0.80')
plt.axhline(1,color='k',lw=.5); plt.xlabel('pas de temps'); plt.ylabel('max |u| (échelle log)'); plt.legend(); plt.grid(alpha=.25); plt.show()

## Fin du premier parcours
Vous avez maintenant suivi le chemin complet : **observation → variable d'état → EDP → source/frontières → discrétisation → schéma numérique → stabilité → expérience**.

Étape suivante possible : reconstruire la condition CFL par une petite analyse de Von Neumann, puis comparer ce toy model à Saint-Venant pour comprendre ce qu'il faut ajouter lorsque l'on veut vraiment modéliser des vagues d'eau.